In [1]:
from copy import deepcopy
import matplotlib.patches as patches
import scipy
import numpy  as np
import math
from scipy.optimize import fsolve
import math

In [2]:
speed_of_light   = scipy.constants.speed_of_light
alpha            = scipy.constants.alpha
epsilon0SI       = scipy.constants.epsilon_0
electronCHARGESI = scipy.constants.elementary_charge
electronMASSSI   = scipy.constants.electron_mass
protonMASSSI     = 2.49e-28#scipy.constants.proton_mass
pion_mass = 139.57061#pion mass in MeV
protonMASS = pion_mass

In [3]:
#original code for trans matix is in BeamLineElement.py 

Debug_bool = True
#ref_part = [x,xp,y,yp,z,E]
#Drift
def drift_trans(Length, ref_part, part):
    p0        = math.sqrt(ref_part[5]**2-pion_mass**2)#math.sqrt(np.dot(iRefPrtcl.getPrOut()[iPrev][:3], iRefPrtcl.getPrOut()[iPrev][:3]))
    E0        = ref_part[5]#iRefPrtcl.getPrOut()[iPrev][3]
    b02       = (p0/E0)**2
    g02       = 1./(1.-b02)
 
    l = Length
    
    TrnsMtrx = np.array( [ \
                          [1.,  l, 0., 0., 0.,        0.], \
                          [0., 1., 0., 0., 0.,        0.], \
                          [0., 0., 1.,  l, 0.,        0.], \
                          [0., 0., 0., 1., 0.,        0.], \
                          [0., 0., 0., 0., 1., l/b02/g02], \
                          [0., 0., 0., 0., 0.,        1.]  \
                         ] )

    return TrnsMtrx

#Aperture
def aperture_trans():
    TrnsMtrx = np.array( [ \
                                  [1., 0., 0., 0., 0., 0.], \
                                  [0., 1., 0., 0., 0., 0.], \
                                  [0., 0., 1., 0., 0., 0.], \
                                  [0., 0., 0., 1., 0., 0.], \
                                  [0., 0., 0., 0., 1., 0.], \
                                  [0., 0., 0., 0., 0., 1.]  \
                                 ] )
    return TrnsMtrx

#FocusQuadrupole
def fquad_trans(ref_part, kFQ, Leng):
    p0        = math.sqrt(ref_part[5]**2-pion_mass**2)#math.sqrt(np.dot(iRefPrtcl.getPrOut()[iPrev][:3], iRefPrtcl.getPrOut()[iPrev][:3]))
    E0        = ref_part[5]#iRefPrtcl.getPrOut()[iPrev][3]
    b0        = p0/E0
    b02       = b0**2
    g02       = 1./(1.-b02)
    
 
    D   = 1.
    Scl = 1.
    if self.getFQmode() == 1:
        D = math.sqrt(1. + 2.*_R[5]/b0 + _R[5]**2)
    else:
        E   = E0 + _R[5]*p0
        p   = math.sqrt(E**2 - protonMASS**2)
        if p > 0:
            Scl = p0 / p
    
    k = kFQ*Scl#self.getkFQ() * Scl
    l = Leng#self.getLength()
    
    b = math.sqrt(k/D)
    a = l * b
    b = b * D
    
    TrnsMtrx = np.array([                                             \
        [   np.cos(a), np.sin(a)/b, 0., 0.,                   0., 0.],\
        [-b*np.sin(a),   np.cos(a), 0., 0.,                   0., 0.],\
        [          0.,          0., np.cosh(a), np.sinh(a)/b, 0., 0.],\
        [          0.,          0., b*np.sinh(a), np.cosh(a), 0., 0.],\
        [          0.,          0.,           0., 0.,  1., l/b02/g02],\
        [          0.,          0.,           0., 0.,  0.,        1.]\
                            ])
                            
    return TrnsMtrx


#DefocusQuadrupole
def dquad_trans(ref_part, kFQ, Leng):
    p0        = math.sqrt(ref_part[5]**2-pion_mass**2)#math.sqrt(np.dot(iRefPrtcl.getPrOut()[iPrev][:3], iRefPrtcl.getPrOut()[iPrev][:3]))
    E0        = ref_part[5]#iRefPrtcl.getPrOut()[iPrev][3]
    b0        = p0/E0
    b02       = b0**2
    g02       = 1./(1.-b02)
    
    D   = 1.
    Scl = 1.
    if self.getDQmode() == 1:
        D = math.sqrt(1. + 2.*_R[5]/b0 + _R[5]**2)
    else:
        E   = E0 + _R[5]*p0
        p   = math.sqrt(E**2 - protonMASS**2)
        if p > 0:
            Scl = p0 / p
    
    k = kFQ*Scl#self.getkFQ() * Scl
    l = Leng#self.getLength()
    
    b = math.sqrt(k/D)
    a = l * b
    b = b * D
    
    TrnsMtrx = np.array([                                               \
        [  np.cosh(a), np.sinh(a)/b,           0.,          0., 0., 0.],\
        [b*np.sinh(a),   np.cosh(a),           0.,          0., 0., 0.],\
        [          0.,           0.,    np.cos(a), np.sin(a)/b, 0., 0.],\
        [          0.,           0., -b*np.sin(a),   np.cos(a), 0., 0.],\
        [          0.,           0.,           0.,  0.,  1., l/b02/g02],\
        [          0.,           0.,           0.,  0.,  0.,        1.]\
    ])

    return TrnsMtrx


#SectorDipole at line 3254
def dipole_trans(Length, Angle, ref_part, part):
    p0        = math.sqrt(ref_part[5]**2-pion_mass**2)#math.sqrt(np.dot(iRefPrtcl.getPrOut()[iPrev][:3], iRefPrtcl.getPrOut()[iPrev][:3]))
    E0        = ref_part[5]#iRefPrtcl.getPrOut()[iPrev][3]
    b0        = p0/E0
    b02       = b0**2
    g02       = 1./(1.-b02)
    
    if Debug_bool:
        print(" Dipole(BeamLineElement).setTransferMatrix:")
        #print("     ----> Reference particle 4-mmtm:",  iRefPrtcl.getPrIn()[0])
        print("         ----> p0, E0:", p0, E0)
        print("     <---- b02, g02:", b02, g02)
    
    E    = E0 + p0*part[5]
    p    = math.sqrt(E**2 - protonMASS**2)
    
    if Debug_bool:
        print("     ----> Particle energy and mmtm:", E, p)
    
    Brho = (1/(speed_of_light*1.E-9))*p/1000.
    rad_angle = Angle*np.pi/180
    r    = Length/rad_angle#Brho / self.getB()
    k0 = 1/r
    c    = np.cos(rad_angle)
    s    = np.sin(rad_angle)
    l    = Length
    
    TrnsMtrx = np.array([
        [     c,            r*s, 0., 0., 0.,                r*(1-c)/b0],
        [  -s/r,              c, 0., 0., 0.,                      s/b0],
        [    0.,             0., 1.,  l, 0.,                        0.],
        [    0.,             0., 0., 1., 0.,                        0.],
        [ -s/b0, -(r/b0)*(1.-c), 0., 0., 1., l/b02/g02 - (l-r*s)/b0**2],
        [    0.,             0., 0., 0., 0.,                        1.]])

    return TrnsMtrx

#SectorDipole
def bdsim_dipole_trans(Length, Angle, ref_part, part, k=1e-10):#k parmeter for quad strength
    p0        = math.sqrt(ref_part[5]**2-pion_mass**2)#math.sqrt(np.dot(iRefPrtcl.getPrOut()[iPrev][:3], iRefPrtcl.getPrOut()[iPrev][:3]))
    E0        = ref_part[5]#iRefPrtcl.getPrOut()[iPrev][3]
    b0        = p0/E0
    b02       = b0**2
    g02       = 1./(1.-b02)
    
    if Debug_bool:
        print(" Dipole(BeamLineElement).setTransferMatrix:")
        #print("     ----> Reference particle 4-mmtm:",  iRefPrtcl.getPrIn()[0])
        print("         ----> p0, E0:", p0, E0)
        print("     <---- b02, g02:", b02, g02)
    
    E    = E0 + p0*part[5]
    p    = math.sqrt(E**2 - protonMASS**2)
    
    if Debug_bool:
        print("     ----> Particle energy and mmtm:", E, p)
    
    Brho = (1/(speed_of_light*1.E-9))*p/1000.
    rad_angle = Angle*np.pi/180
    r    = Length/rad_angle#Brho / self.getB()
    l    = Length
    k0 = 1/r

    k_0 = 1/r
    K_x = k+k_0**2
    K_y = k
    theta_x = np.sqrt(K_x)*l
    theta_y = np.sqrt(K_y)*l
    
    c_x    = np.cosh(theta_x)
    s_x    = np.sinh(theta_x)
    c_y    = np.cos(theta_y)
    s_y    = np.sin(theta_y)
    
    TrnsMtrx = np.array([
        [     c_x,s_x/np.sqrt(K_x), 0., 0., 0., (1-c_x)/(b0*np.sqrt(K_x))],
        [  s_x*np.sqrt(K_x),   c_x, 0., 0., 0.,                    s_x/b0],
        [    0.,0., c_y,  s_y/np.sqrt(K_y), 0.,                        0.],
        [    0.,0.,-np.sqrt(K_y)*s_y,  c_y, 0.,                        0.],
        [-s_x/b0,-1/(b0*np.sqrt(K_x))*(1.-c_x), 0., 0., 1.,l/b02/g02 - (l-s_x/np.sqrt(K_x))/b0**2],
        [    0.,             0., 0., 0., 0.,                        1.]])

    return TrnsMtrx


#ref_part = [x,xp,y,yp,z,E]
def get_delta(ref_part, part):
    p0 = math.sqrt(ref_part[5]**2-pion_mass**2)
    delta_E = part[5]-ref_part[5]
    return delta_E/p0

def get_tr_space(ref_part, part):
    part_tr = [0,0,0,0,0,0]
    delta = get_delta(ref_part, part)
    for i in range(0,5):
        part_tr[i] = part[i]

    part_tr[5] = delta

    return part_tr

def test_sympl_2d(trans_2d): #
    J = np.array([[0,1],[-1,0]])
    
    symple_mat = (trans_2d.T)@J@trans_2d

    return symple_mat

In [4]:
#ref_part = [x,xp,y,yp,z,E]

ref_part_tr_space = [0, 0, 0, 0, 0, 5000]
part_val = [0.1, 0, 0, 0, 0, 5500]
delta = get_delta(ref_part_tr_space, part_val)
print("delta : "+str(delta))
part_tr_space = get_tr_space(ref_part_tr_space, part_val)#[0, 0, 0, 0, 0, delta]
tr_mtrx = dipole_trans(1,5,ref_part_tr_space,part_tr_space)#dipole_trans
print(np.array2string(tr_mtrx, precision=6, suppress_small=True))#
print(np.array2string(test_sympl_2d(np.array([[tr_mtrx[0][0], tr_mtrx[0][1]],[tr_mtrx[1][0], tr_mtrx[1][1]]])), precision=6, suppress_small=True))
part_tr_space_next = np.dot(tr_mtrx, part_tr_space)
print(part_tr_space_next)

delta : 0.1000389826932649
 Dipole(BeamLineElement).setTransferMatrix:
         ----> p0, E0: 4998.051624865857 5000
     <---- b02, g02: 0.9992208017929692 1283.370509553188
     ----> Particle energy and mmtm: 5500.0 5498.228809791771
[[ 0.996195  0.998731  0.        0.        0.        0.043623]
 [-0.007606  0.996195  0.        0.        0.        0.08719 ]
 [ 0.        0.        1.        1.        0.        0.      ]
 [ 0.        0.        0.        1.        0.        0.      ]
 [-0.08719  -0.043623  0.        0.        1.       -0.00049 ]
 [ 0.        0.        0.        0.        0.        1.      ]]
[[ 0.  1.]
 [-1.  0.]]
[ 0.10398342  0.00796179  0.          0.         -0.00876798  0.10003898]


In [5]:
tr_mtrx = dipole_trans(1,5,ref_part_tr_space,part_tr_space)#dipole_trans
print(np.array2string(test_sympl_2d(np.array([[tr_mtrx[0][0], tr_mtrx[0][1]],[tr_mtrx[1][0], tr_mtrx[1][1]]])), precision=6, suppress_small=True))

 Dipole(BeamLineElement).setTransferMatrix:
         ----> p0, E0: 4998.051624865857 5000
     <---- b02, g02: 0.9992208017929692 1283.370509553188
     ----> Particle energy and mmtm: 5500.0 5498.228809791771
[[ 0.  1.]
 [-1.  0.]]


In [6]:
tr_mtrx = bdsim_dipole_trans(1,5,ref_part_tr_space,part_tr_space, 1e-1)#dipole_trans
print(np.array2string(test_sympl_2d(np.array([[tr_mtrx[0][0], tr_mtrx[0][1]],[tr_mtrx[1][0], tr_mtrx[1][1]]])), precision=6, suppress_small=True))

 Dipole(BeamLineElement).setTransferMatrix:
         ----> p0, E0: 4998.051624865857 5000
     <---- b02, g02: 0.9992208017929692 1283.370509553188
     ----> Particle energy and mmtm: 5500.0 5498.228809791771
[[ 0.  1.]
 [-1.  0.]]


In [7]:
print(np.array([[tr_mtrx[0][0], tr_mtrx[0][1]],[tr_mtrx[1][0], tr_mtrx[1][1]]]))
print(np.array([[tr_mtrx[0][0], tr_mtrx[0][1]],[tr_mtrx[1][0], tr_mtrx[1][1]]]).T)

[[1.054292   1.01803266]
 [0.10955603 1.054292  ]]
[[1.054292   0.10955603]
 [1.01803266 1.054292  ]]


In [8]:
#ref_part = [x,xp,y,yp,z,E]

ref_part_tr_space = [0, 0, 0, 0, 0, 5000]
part_val = [0.1, 0.01, 0.1, 0, 0, 5500]
delta = get_delta(ref_part_tr_space, part_val)
print("delta : "+str(delta))
part_tr_space = get_tr_space(ref_part_tr_space, part_val)#[0, 0, 0, 0, 0, delta]
tr_mtrx = bdsim_dipole_trans(1,5,ref_part_tr_space,part_tr_space, 1e-1)
print(tr_mtrx)
part_tr_space_next = np.dot(tr_mtrx, part_tr_space)
print(part_tr_space_next)

delta : 0.1000389826932649
 Dipole(BeamLineElement).setTransferMatrix:
         ----> p0, E0: 4998.051624865857 5000
     <---- b02, g02: 0.9992208017929692 1283.370509553188
     ----> Particle energy and mmtm: 5500.0 5498.228809791771
[[ 1.054292    1.01803266  0.          0.          0.         -0.16556472]
 [ 0.10955603  1.054292    0.          0.          0.          0.33409368]
 [ 0.          0.          0.95041528  0.98341647  0.          0.        ]
 [ 0.          0.         -0.09834165  0.95041528  0.          0.        ]
 [-0.33409368  0.16556472  0.          0.          1.          0.01882653]
 [ 0.          0.          0.          0.          0.          1.        ]]
[ 0.0990466   0.05492091  0.09504153 -0.00983416 -0.02987033  0.10003898]


In [9]:
detTrnsfrMtrx = np.linalg.det(tr_mtrx)
error         = abs(1. - abs(detTrnsfrMtrx))
print(error)

0.0
